In [27]:
# import necessary libraries
import pandas as pd
import numpy as np
import re
from pathlib import Path

# Define project root dynamically
PROJECT_ROOT = Path.cwd()

# If notebook is inside 'notebooks', move one level up
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

# Load the dataset
file_path = PROJECT_ROOT / "data" / "anonymized_data" / "customer_feedback_anonymized.csv"

df = pd.read_csv(file_path, encoding="ISO-8859-1")

print("Loaded from:", file_path)
print("Shape:", df.shape)



Loaded from: c:\Users\LENOVO\OneDrive\Documents\Strath\Masters\sentiment-analysis-tool\data\anonymized_data\customer_feedback_anonymized.csv
Shape: (89942, 5)


In [28]:

from pathlib import Path

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

PROCESSED_PATH = PROJECT_ROOT / "data" / "processed" / "customer_feedback_cleaned.csv"

PROCESSED_PATH.parent.mkdir(parents=True, exist_ok=True)
df.to_csv(PROCESSED_PATH, index=False)

print("Cleaned data saved to:", PROCESSED_PATH)

# Remove duplicates
initial_rows = df.shape[0]

df = df.drop_duplicates()

print("Duplicates removed:", initial_rows - df.shape[0])
print("New shape:", df.shape)


# Check for missing values
df.isnull().sum()

df = df.dropna(subset=["text"])


# Text normalization
df["text"] = df["text"].astype(str)

df["text"] = df["text"].str.lower()

# Remove URLs
df["text"] = df["text"].apply(
    lambda x: re.sub(r"http\S+|www\S+", "", x)
)

# Remove email artifacts
df["text"] = df["text"].apply(
    lambda x: re.sub(r"\S+@\S+", "", x)
)
# Remove special characters and numbers
df["text"] = df["text"].apply(
    lambda x: re.sub(r"[^a-zA-Z\s]", "", x)
)   

# Remove extra whitespace
df["text"] = df["text"].apply(
    lambda x: re.sub(r"\s+", " ", x).strip()
)   

#Remove very short reviews (less than 3 words)
df = df[df["text"].str.split().str.len() > 3]
print("Shape after removing short reviews:", df.shape)

# Save the cleaned dataset
PROCESSED_PATH.parent.mkdir(parents=True, exist_ok=True)

df.to_csv(PROCESSED_PATH, index=False)

print("Cleaned data saved to:", PROCESSED_PATH)



Cleaned data saved to: c:\Users\LENOVO\OneDrive\Documents\Strath\Masters\sentiment-analysis-tool\data\processed\customer_feedback_cleaned.csv
Duplicates removed: 6750
New shape: (83192, 5)
Shape after removing short reviews: (76147, 5)
Cleaned data saved to: c:\Users\LENOVO\OneDrive\Documents\Strath\Masters\sentiment-analysis-tool\data\processed\customer_feedback_cleaned.csv


In [29]:
# Remove specific words (e.g., "name", "nsb") from the reviews
words_to_remove = ["name", "nbsb"]

pattern = r'\b(?:' + '|'.join(words_to_remove) + r')\b'

df["text"] = df["text"].str.replace(pattern, "", regex=True)

# Normalize spaces again
df["text"] = df["text"].str.replace(r"\s+", " ", regex=True).str.strip()



In [30]:
# Clean up extra whitespace after removing words
df["text"] = df["text"].str.replace("\xa0", " ", regex=False)
df["text"] = df["text"].str.replace("&nbsp;", " ", regex=False)

df["text"] = df["text"].str.replace(r"\s+", " ", regex=True).str.strip()


In [31]:
df.sample(10)


,source,date,channel,product,text
26986,CRM Tool,2025-11-14 00:00:00,Service Email,Accounts,nbsp to the chairmannbsp united nationsnbsp co...
62758,CRM Tool,2025-11-27 00:00:00,Service Web,Motor Vehicle,team one month from extension proceed from exp...
52918,CRM Tool,2025-11-26 00:00:00,Service Web,Airtime Top Up,support with the reversal of of airtime errone...
48561,CRM Tool,2025-11-25 00:00:00,Service Email,Customer Service,to the chairmannbsp nbspunited nations council...
65505,CRM Tool,2025-12-02 00:00:00,Service Web,Physical Security,attached branch snap check findings on physica...
54466,CRM Tool,2025-11-27 00:00:00,Service Email,EazzyBiz,namenbspname find the below invoice of usd to ...
8504,CRM Tool,2025-11-06 00:00:00,Service Web,Equity Mobile,do i reset my forgotten password
49343,CRM Tool,2025-11-24 00:00:00,Service Email,Equity Mobile,currently experiencing challenges logging into...
43611,CRM Tool,2025-11-21 00:00:00,Service Email,EazzyBiz,share the transaction reference namenbspname t...
65195,CRM Tool,2025-12-02 00:00:00,Service Email,Customer Support,i wish to transfer from our above quoted accou...


In [32]:
# Language detection
from langdetect import detect
from langdetect.lang_detect_exception import LangDetectException

def detect_language(text):
    try:
        return detect(text)
    except LangDetectException:
        return "unknown"

df["language"] = df["text"].apply(detect_language)


In [33]:
# check language distribution
df["language"].value_counts()


language
en         72443
no           469
fr           443
it           358
af           309
ca           230
sv           229
pt           211
da           202
nl           191
ro           171
sw           148
id           105
sl            89
tl            86
et            74
cy            62
es            49
de            48
fi            47
unknown       37
pl            34
hr            23
so            22
sk            18
cs            15
sq            14
tr             7
lv             5
lt             3
vi             3
hu             2
Name: count, dtype: int64

In [34]:
# Save the cleaned dataset after removing specific words
from pathlib import Path


# Define Project Root

PROJECT_ROOT = Path.cwd()

# If running from notebooks folder, move one level up
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent


# Define Save Path

FINAL_PATH = PROJECT_ROOT / "data" / "processed" / "customer_feedback_cleaned_for_annotation.csv"

# Create folder if it doesn't exist
FINAL_PATH.parent.mkdir(parents=True, exist_ok=True)


# Save File

df.to_csv(FINAL_PATH, index=False, encoding="utf-8")

print("Final cleaned dataset saved to:")
print(FINAL_PATH)
print("Final dataset shape:", df.shape)


Final cleaned dataset saved to:
c:\Users\LENOVO\OneDrive\Documents\Strath\Masters\sentiment-analysis-tool\data\processed\customer_feedback_cleaned_for_annotation.csv
Final dataset shape: (76147, 6)


In [36]:
# Remove unwanted tokens
df['text'] = df['text'].str.replace(r'\b(name|nbsp|equity)\b', '', 
                                    regex=True, case=False)

# Remove extra spaces
df['text'] = df['text'].str.replace(r'\s+', ' ', regex=True).str.strip()


# Remove 'equity' from entire dataframe
for col in df.select_dtypes(include='object').columns:
    df[col] = df[col].str.replace(r'\bequity\b', '', regex=True, case=False)

for col in df.select_dtypes(include='object').columns:
    df[col] = df[col].str.replace(r'\s+', ' ', regex=True).str.strip()


C:\Users\LENOVO\AppData\Local\Temp\ipykernel_22136\3246192969.py:10: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  for col in df.select_dtypes(include='object').columns:
C:\Users\LENOVO\AppData\Local\Temp\ipykernel_22136\3246192969.py:13: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_gu

In [37]:
# Detect mixed languages

# Define sets of common keywords for English and Swahili
english_words = {"the", "and", "is", "are", "bank", "account", "loan"}
swahili_words = {"na", "ni", "kwa", "hii", "sasa", "lakini", "mimi"}

# Function to detect language based on keyword presence
def detect_language(text):
    words = set(text.lower().split())
    
    eng = len(words.intersection(english_words))
    swa = len(words.intersection(swahili_words))
    
    if eng > 0 and swa > 0:
        return "sheng"
    elif eng > 0:
        return "english"
    elif swa > 0:
        return "swahili"
    else:
        return "unknown"

df["language"] = df["text"].apply(detect_language)


In [39]:
# Define Save Path

FINAL_PATH = PROJECT_ROOT / "data" / "processed" / "cleaned_final_dataset.csv"

# Create folder if it doesn't exist
FINAL_PATH.parent.mkdir(parents=True, exist_ok=True)


# Save File

df.to_csv(FINAL_PATH, index=False, encoding="utf-8")

print("Final cleaned dataset saved to:")
print(FINAL_PATH)
print("Final dataset shape:", df.shape)


Final cleaned dataset saved to:
c:\Users\LENOVO\OneDrive\Documents\Strath\Masters\sentiment-analysis-tool\data\processed\cleaned_final_dataset.csv
Final dataset shape: (76147, 6)
